In [ ]:
import xarray as xr

In [ ]:
# ANCILLARY FILE FOR HIMAWARI CMIC
ancil = xr.open_dataset('/g/data/ra22/satellite-products/arc/obs/himawari-ahi/fldk/latest/ancillary/00000000000000-P1S-ABOM_GEOM_SENSOR-PRJ_GEOS141_2000-HIMAWARI8-AHI.nc')
ancil = ancil.isel(time=0)

In [ ]:
# LOAD HIMAWARI CMIC DATA
file_path = Path(f'/g/data/rv74/satellite-products/arc/der/himawari-ahi/cloud/cmic/latest/{year}/{month:02d}/{day}/')
files = [f for f in file_path.glob(f"*{year}{month:02d}{day:02d}T{hour:02d}00*.nc")]

# Pre-load ancillary coordinates once
lat_ancil = ancil.lat.data
lon_ancil = ancil.lon.data

# DATETIME IS NOT IN NETCDF FILE, HAS TO BE ADDED WHILE OPENING FILE FROM THE FILE NAME
def make_preprocess_func(filepaths):
    def _preprocess(ds, file_idx):
        f = filepaths[file_idx]
        match = re.search(r"_(\d{8}T\d{6})Z\.nc$", f.name)
        time = pd.to_datetime(match.group(1), format="%Y%m%dT%H%M%S")
        ds = ds.assign_coords(
            time=time,
            lat=(("ny", "nx"), lat_ancil),
            lon=(("ny", "nx"), lon_ancil)
        )
        ds["cmic_lwp"] = ds["cmic_lwp"].fillna(0)
        return ds

    def preprocess_with_index(i, ds):
        return _preprocess(ds, i)

    return preprocess_with_index

preprocess_func = make_preprocess_func(files)

# OPEN THE FILES
datasets = []
for i, f in enumerate(files):
    ds = xr.open_dataset(f, engine='h5netcdf', chunks='auto')
    ds = preprocess_func(i, ds)
    datasets.append(ds)
him_cmic = xr.concat(datasets, dim="time")

# SELECT JUST THE DATA OF INTEREST
him_cmic_slice = him_cmic.sel(ny=slice(-1.1e6, -4.25e6), nx=slice(-3e6, 1.3e6))
him_wp = him_cmic_slice[['cmic_lwp', 'cmic_iwp']]

# Define region bounds
lat_min, lat_max = -44.5, -10.0
lon_min, lon_max = 112.0, 156.126

# Define target 1D grid
target = xr.Dataset({
    "lat": (["lat"], np.linspace(lat_min, lat_max, barra_lwp.lat.shape[0])),
    "lon": (["lon"], np.linspace(lon_min, lon_max, barra_lwp.lon.shape[0])),
})

regridder = xe.Regridder(him_wp, target, "bilinear")
him_wp_reg = regridder(him_wp)

him_wp_reg = him_wp_reg.interp(
    lat=barra_lwp.lat,
    lon=barra_lwp.lon,
    method='linear'
)

# LOAD DATA TO PREVENT LATER DASK ISSUES
him_wp_reg = him_wp_reg.load()